In [1]:
import yfinance
import numpy as np
import pandas as pd
from tqdm import tqdm

## Sample

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import json
import time
import os
from datetime import datetime

def ticker_to_json(ticker_symbol, outdir="./yf_dumps", sleep_sec=0.5):
    """
    Extracts all accessible data from a yfinance.Ticker object and dumps to JSON.
    Returns the combined dictionary (also written to disk).
    """
    os.makedirs(outdir, exist_ok=True)
    t = yf.Ticker(ticker_symbol)
    dump = {"ticker": ticker_symbol.upper(), "asof_utc": datetime.now().isoformat()}

    # Simple helper to convert DataFrames safely
    def safe_df(obj):
        if isinstance(obj, pd.DataFrame):
            obj.columns = [str(col) for col in obj.columns]
            return obj.reset_index().to_dict(orient="records")
        elif isinstance(obj, pd.Series):
            return obj.to_dict()
        elif obj is None:
            return None
        else:
            return str(obj)

    def convert_dict_keys_to_str(d):
        return {k.strftime("%Y-%m-%d"): v for k, v in d.items()}

    # --- Try each attribute defensively ---
    ticker_attr = ['actions', 'balance_sheet', 'capital_gains', 'cash_flow', 'earnings_dates', 'financials', 'history_metadata', 'income_stmt', 'info', 
    'news', 'mutualfund_holders', 'major_holders', 'isin', 'options', 'quarterly_balance_sheet', 'quarterly_cash_flow', 'quarterly_financials', 
    'quarterly_income_stmt', 'sec_filings', 'splits']

    #conversion_list = ['balance_sheet', 'cash_flow', 'financials', 'income_stmt', 'quarterly_balance_sheet', 'quarterly_cash_flow', 'quarterly_financials', 'quarterly_income_stmt']
    for a in ticker_attr:
        try:
            val = getattr(t, a)
            val = safe_df(val)
            if type(val) == dict:
                val = convert_dict_keys_to_str(val)
            dump[a] = val
        except Exception as e:
            dump[a] = f"Error: {type(e).__name__}: {e}"

    # --- Option chains (can be large, skip or limit) ---
    option_dates = []
    try:
        option_dates = t.options or []
    except Exception:
        option_dates = []

    chains = {}
    for d in option_dates[:3]:  # limit to first 3 expiries to avoid massive files
        try:
            oc = t.option_chain(d)
            chains[d] = {
                "calls": safe_df(oc.calls),
                "puts": safe_df(oc.puts)
            }
            time.sleep(0.2)
        except Exception as e:
            chains[d] = f"Error: {type(e).__name__}: {e}"
    dump["option_chains"] = chains

    # --- Recent price history (1mo daily, 5y weekly, 10y monthly) ---
    try:
        dump["history_5y_daily"] = safe_df(t.history(period="5y", interval="1d"))
        dump["history_5y_weekly"] = safe_df(t.history(period="5y", interval="1wk"))
        dump["history_10y_monthly"] = safe_df(t.history(period="10y", interval="1mo"))
    except Exception as e:
        dump["history_error"] = str(e)

    outpath = os.path.join(outdir, f"{ticker_symbol.upper()}_dump.json")
    with open(outpath, "w", encoding="utf-8") as f:
        json.dump(dump, f, indent=2, default=str)

    print(f"[{ticker_symbol}] written to {outpath}")
    time.sleep(sleep_sec)
    return dump

def json_loader(json_path):
    """
    Loads a JSON dump created by ticker_to_json.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

In [3]:
data_path = "./yf_dumps"

In [4]:
existing_json = [file for file in  os.listdir(data_path) if file.endswith(".json")]
existing_ticker = [file.replace("_dump.json", "") for file in existing_json]

# Load metadata and prepare ticker list
metadata = pd.read_csv('data/metadata.csv', encoding='cp1252')
ticker_list = metadata.Symbol.values.tolist()
print(len(ticker_list))
replace = False 

if not replace: 
    ticker_list = [t for t in ticker_list if t not in existing_ticker]

print(f"Total tickers to process: {len(ticker_list)}")
print(f'Total existing tickers: {len(existing_ticker)}')

503
Total tickers to process: 0
Total existing tickers: 503


In [5]:
for ticker in tqdm(ticker_list):
    dump = ticker_to_json(ticker, outdir=data_path, sleep_sec=3)


0it [00:00, ?it/s]


In [6]:
data = {}
for file in existing_json:
    data[file.replace("_dump.json", "")] = json_loader(os.path.join(data_path, file))
    
price_history = list() 
for ticker in data.keys():
    df = pd.DataFrame(data[ticker]["history_5y_daily"])
    df["ticker"] = data[ticker]["ticker"]
    price_history.append(df)

price_history_df = pd.concat(price_history, ignore_index=True)


In [7]:
price_history_df.columns = [col.lower().replace(" ", "_") for col in price_history_df.columns]

In [8]:
price_history_df.to_csv("data/price_history_5y_daily.csv", index=False)

## Test Dumping Individual Stock Component in JSON

In [ ]:
test_dump = False

In [ ]:
if test_dump:
    ticker_symbol = 'MMM'
    outdir = './Test'
    sleep_sec = 0.5

    dump = json_loader(os.path.join(outdir, f"{ticker_symbol.upper()}_dump.json"))
    for k in dump.keys():
        print(k)
        outpath = os.path.join(outdir, f"{ticker_symbol.upper()}_dump_{k}.json")
        with open(outpath, "w", encoding="utf-8") as f:
            json.dump(dump[k], f, indent=2, default=str)

        print(f"[{ticker_symbol}] written to {outpath}")
        time.sleep(sleep_sec)

ticker
[MMM] written to ./Test\MMM_dump_ticker.json
asof_utc
[MMM] written to ./Test\MMM_dump_asof_utc.json
actions
[MMM] written to ./Test\MMM_dump_actions.json
balance_sheet
[MMM] written to ./Test\MMM_dump_balance_sheet.json
capital_gains
[MMM] written to ./Test\MMM_dump_capital_gains.json
cash_flow
[MMM] written to ./Test\MMM_dump_cash_flow.json
earnings_dates
[MMM] written to ./Test\MMM_dump_earnings_dates.json
financials
[MMM] written to ./Test\MMM_dump_financials.json
history_metadata
[MMM] written to ./Test\MMM_dump_history_metadata.json
income_stmt
[MMM] written to ./Test\MMM_dump_income_stmt.json
info
[MMM] written to ./Test\MMM_dump_info.json
news
[MMM] written to ./Test\MMM_dump_news.json
mutualfund_holders
[MMM] written to ./Test\MMM_dump_mutualfund_holders.json
major_holders
[MMM] written to ./Test\MMM_dump_major_holders.json
isin
[MMM] written to ./Test\MMM_dump_isin.json
options
[MMM] written to ./Test\MMM_dump_options.json
quarterly_balance_sheet
[MMM] written to ./Tes

### Creating Balance Sheet DF

In [94]:
from collections import Counter

class BalanceSheetJSONPreprocessor:
    def __init__(self, file_names, data_path="./yf_dumps"):
        self.file_names = file_names
        self.data_path = data_path
        self.data_raw = dict()
        self.load_status = dict()
        self.preprocessed_df = dict()
        self.counters = dict()
        self.column_status = dict()
        
        for name in self.file_names:
            stock_raw = json_loader(self.data_path + "/" + name)
            ticker =  name.replace(self.data_path + "/", "").replace("_dump.json", "")
            self.data_raw[ticker] = stock_raw
            
    def preprocess_to_df(
            self, 
            data_key
        ):
        df_list = list()
        status_dict = dict()
        self._init_counters(data_key)
        
        for ticker, stock_raw in self.data_raw.items():
            if data_key in stock_raw.keys():
                if type(stock_raw[data_key]) is list:
                    if len(stock_raw[data_key]) > 0:
                        df_in = pd.DataFrame(stock_raw[data_key]).set_index('index').T.reset_index().rename(columns={'index': 'date'})
                        df_in['statement_type'] = 'quarterly'
                        df_in['ticker'] = ticker
                        df_list.append(df_in)
                        
                        self._update_counters(data_key, df_in)
                        status_dict[ticker] = f'{data_key} load success'
                    else:
                        status_dict[ticker] = f'{data_key} list empty'
                else:
                    status_dict[ticker] = f'{data_key} not list'
            else:
                status_dict[ticker] = f'{data_key} missing'
            
        self.load_status[data_key] = pd.DataFrame(status_dict.items(), columns=['ticker', 'status'])
        self.preprocessed_df[data_key] = pd.concat(df_list, ignore_index=True)
        self.column_status[data_key] = (
                    pd.DataFrame({
                        "num_df_present": self.counters[data_key]['column_presence'],
                        "num_row_present": self.counters[data_key]['column_rows'],
                        "num_row_null": self.counters[data_key]['column_nulls'],
                    })
                    .fillna(0)
                    .astype(int)
                    .sort_values("num_df_present", ascending=False)
                )
        
    def preprocess_batch(self, data_keys):
        for key in data_keys:
            self.preprocess_to_df(key)
    
    def combine_all_dfs(self):
        combined_df_list = list()
        for key, df in self.preprocessed_df.items():
            df['load_key'] = key
            combined_df_list.append(df)
        preprocessed_df = pd.concat(combined_df_list, ignore_index=True)
        
        columns_status_list = list()
        for key, df in self.column_status.items():
            df['load_key'] = key
            columns_status_list.append(df.reset_index().rename(columns={'index': 'column_name'}))
        column_status = pd.concat(columns_status_list, ignore_index=True)
        
        load_status_list = list()
        for key, df in self.load_status.items():
            df['load_key'] = key
            load_status_list.append(df) 
        load_status = pd.concat(load_status_list, ignore_index=True)
        
        return preprocessed_df, column_status, load_status
            

    def _init_counters(self, key):
        self.counters[key] = dict()
        self.counters[key]['column_presence'] = Counter()
        self.counters[key]['column_nulls'] = Counter()
        self.counters[key]['column_rows'] = Counter()
    
    def _update_counters(self, key, df):
        cols = df.columns.tolist()
        null_counts = df.isna().sum()
        self.counters[key]['column_presence'].update(cols)
        self.counters[key]['column_nulls'].update(null_counts.to_dict())
        self.counters[key]['column_rows'].update({c: len(df) for c in cols})


In [95]:
preprocessor = BalanceSheetJSONPreprocessor(existing_json, data_path=data_path)
preprocessor.preprocess_batch(
    [
        'quarterly_balance_sheet', 
        'balance_sheet', 
        'cash_flow',
        'quarterly_cash_flow',
        'financials',
        'quarterly_financials',
        'income_stmt',
        'quarterly_income_stmt'
     ]
)

In [96]:
preprocessed_df, column_status, load_status = preprocessor.combine_all_dfs()

C:\Users\wongs\AppData\Local\Temp\ipykernel_24932\2703700253.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['load_key'] = key
C:\Users\wongs\AppData\Local\Temp\ipykernel_24932\2703700253.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['load_key'] = key
C:\Users\wongs\AppData\Local\Temp\ipykernel_24932\2703700253.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.con

In [98]:
preprocessed_df

index,date,Ordinary Shares Number,Share Issued,Net Debt,Total Debt,Tangible Book Value,Invested Capital,Working Capital,Net Tangible Assets,Common Stock Equity,...,Rent And Landing Fees,Occupancy And Equipment,Professional Expense And Contract Services Expense,Other Non Interest Expense,Insurance And Claims,Excise Taxes,Depletion Income Statement,Net Income From Tax Loss Carryforward,Net Income Extraordinary,Securities Amortization
0,2025-09-30 00:00:00,1.477326e+10,1.477326e+10,6.272300e+10,9.865700e+10,7.373300e+10,1.723900e+11,-1.767400e+10,7.373300e+10,7.373300e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-06-30 00:00:00,1.485672e+10,1.485672e+10,6.542900e+10,1.016980e+11,6.583000e+10,1.675280e+11,-1.862900e+10,6.583000e+10,6.583000e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-03-31 00:00:00,1.493932e+10,1.493932e+10,7.002400e+10,9.818600e+10,6.679600e+10,1.649820e+11,-2.589700e+10,6.679600e+10,6.679600e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-12-31 00:00:00,1.503787e+10,1.503787e+10,6.650000e+10,9.679900e+10,6.675800e+10,1.635570e+11,-1.112500e+10,6.675800e+10,6.675800e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-09-30 00:00:00,1.511679e+10,1.511679e+10,7.668600e+10,1.066290e+11,5.695000e+10,1.635790e+11,-2.340500e+10,5.695000e+10,5.695000e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22293,2025-06-30 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22294,2025-03-31 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22295,2024-12-31 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22296,2024-09-30 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [99]:
column_status.to_csv("eda/column_status.csv", index=False)

In [14]:
from typing import Dict
import pandas as pd

def compare_dataframe_columns(dfs: Dict[str, pd.DataFrame]):
    """
    dfs: dict of {name: DataFrame}
    
    Returns a dict with:
      - per-DF unique columns
      - per-DF overlapping columns
      - columns common to all DFs
      - columns present in at least 2 DFs
    """
    # Column sets per DF
    col_sets = {name: set(df.columns) for name, df in dfs.items()}
    
    # Union and intersection
    all_cols = set.union(*col_sets.values())
    common_all = set.intersection(*col_sets.values())
    
    # Count presence
    col_counts = {}
    for cols in col_sets.values():
        for c in cols:
            col_counts[c] = col_counts.get(c, 0) + 1
    
    common_some = {c for c, n in col_counts.items() if n >= 2}
    
    # Per-DF comparison
    per_df = {}
    for name, cols in col_sets.items():
        other_cols = set.union(*(v for k, v in col_sets.items() if k != name))
        per_df[name] = {
            "unique": cols - other_cols,
            "overlapping": cols & other_cols,
            "missing_from_others": all_cols - cols
        }
    
    return {
        "per_dataframe": per_df,
        "common_to_all": common_all,
        "common_to_at_least_two": common_some,
        "all_columns": all_cols
    }


In [15]:
balancesheet_df_all = pd.concat(balancesheet_list, ignore_index=True)
cashflow_df_all = pd.concat(cashflow_list, ignore_index=True)
income_df_all = pd.concat(income_list, ignore_index=True)
financials_df_all = pd.concat(financials_list, ignore_index=True)

In [16]:
result = compare_dataframe_columns({
    "balancesheet_df_all": balancesheet_df_all,
    "cashflow_df_all": cashflow_df_all,
    "income_df_all": income_df_all,
    "financials_df_all": financials_df_all
})


In [17]:
result

{'per_dataframe': {'balancesheet_df_all': {'unique': {'accounts_payable',
    'accounts_receivable',
    'accrued_interest_receivable',
    'accumulated_depreciation',
    'additional_paid_in_capital',
    'allowance_for_doubtful_accounts_receivable',
    'assets_held_for_sale_current',
    'available_for_sale_securities',
    'buildings_and_improvements',
    'capital_lease_obligations',
    'capital_stock',
    'cash_and_cash_equivalents',
    'cash_cash_equivalents_and_federal_funds_sold',
    'cash_cash_equivalents_and_short_term_investments',
    'cash_equivalents',
    'cash_financial',
    'commercial_paper',
    'common_stock',
    'common_stock_equity',
    'construction_in_progress',
    'current_accrued_expenses',
    'current_assets',
    'current_capital_lease_obligation',
    'current_debt',
    'current_debt_and_capital_lease_obligation',
    'current_deferred_assets',
    'current_deferred_liabilities',
    'current_deferred_revenue',
    'current_deferred_taxes_assets'

In [18]:
for name, info in result["per_dataframe"].items():
    print(f"\n{name}")
    print("  unique:", sorted(info["unique"]))
    print("  overlapping:", sorted(info["overlapping"]))


balancesheet_df_all
  unique: ['accounts_payable', 'accounts_receivable', 'accrued_interest_receivable', 'accumulated_depreciation', 'additional_paid_in_capital', 'allowance_for_doubtful_accounts_receivable', 'assets_held_for_sale_current', 'available_for_sale_securities', 'buildings_and_improvements', 'capital_lease_obligations', 'capital_stock', 'cash_and_cash_equivalents', 'cash_cash_equivalents_and_federal_funds_sold', 'cash_cash_equivalents_and_short_term_investments', 'cash_equivalents', 'cash_financial', 'commercial_paper', 'common_stock', 'common_stock_equity', 'construction_in_progress', 'current_accrued_expenses', 'current_assets', 'current_capital_lease_obligation', 'current_debt', 'current_debt_and_capital_lease_obligation', 'current_deferred_assets', 'current_deferred_liabilities', 'current_deferred_revenue', 'current_deferred_taxes_assets', 'current_deferred_taxes_liabilities', 'current_liabilities', 'current_notes_payable', 'current_provisions', 'defined_pension_benefit

In [ ]:
company_data = balancesheet_df_all.merge(
    cashflow_df_all,
    on=['ticker', 'date', 'statement_type'],
    how='outer',
).merge(
    income_df_all,
    on=['ticker', 'date', 'statement_type'],
    how='outer',
)
company_data = company_data.drop(columns=['statement_type']).drop_duplicates()
company_data.to_csv('./data/company_financials.csv', index=False)